In [1]:
# -*- coding: utf-8 -*-
import torch
from qwen_vl_utils import process_vision_info
from transformers import AutoProcessor
from vllm import LLM, SamplingParams

import os
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'

def prepare_inputs_for_vllm(messages, processor):
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    # qwen_vl_utils 0.0.14+ reqired
    image_inputs, video_inputs, video_kwargs = process_vision_info(
        messages,
        image_patch_size=processor.image_processor.patch_size,
        return_video_kwargs=True,
        return_video_metadata=True
    )
    print(f"video_kwargs: {video_kwargs}")

    mm_data = {}
    if image_inputs is not None:
        mm_data['image'] = image_inputs
    if video_inputs is not None:
        mm_data['video'] = video_inputs

    return {
        'prompt': text,
        'multi_modal_data': mm_data,
        'mm_processor_kwargs': video_kwargs
    }


/home/zechuan/miniconda3/envs/vllm/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [9]:

if __name__ == '__main__':
    # messages = [
    #     {
    #         "role": "user",
    #         "content": [
    #             {
    #                 "type": "video",
    #                 "video": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen2-VL/space_woaudio.mp4",
    #             },
    #             {"type": "text", "text": "这段视频有多长"},
    #         ],
    #     }
    # ]
    prompt = "图片如果包含任何文本或表格数据，生成简短的描述，概述其关键信息；如果没有，请返回 '无描述'。"
    messages = [
        {
            "role": "user",
            "content": [
              {
                  "type": "image",
                  "image": "/home/zechuan/m3docrag/contents/visdom_test_output_text/dots_ocr_text_test/2024_Tencent_ESG/2024_Tencent_ESG_page_17_img_17.png",
              },
              {"type": "text", "text": prompt},
            ],
        }
    ]

    # TODO: change to your own checkpoint path
    checkpoint_path = "/mnt/SSD2_8TB/zechuan/huggingface/models--Qwen--Qwen3-VL-4B-Instruct/snapshots/ebb281ec70b05090aa6165b016eac8ec08e71b17"
    processor = AutoProcessor.from_pretrained(checkpoint_path)
    inputs = [prepare_inputs_for_vllm(message, processor) for message in [messages]]

    llm = LLM(
        model=checkpoint_path,
        trust_remote_code=True,
        gpu_memory_utilization=0.5,
        enforce_eager=False,
        seed=0,
        max_model_len=32768
    )

    sampling_params = SamplingParams(
        temperature=0,
        max_tokens=1024,
        top_k=-1,
        stop_token_ids=[],
    )

    for i, input_ in enumerate(inputs):
        print()
        print('=' * 40)
        print(f"Inputs[{i}]: {input_['prompt']=!r}")
    print('\n' + '>' * 40)

    outputs = llm.generate(inputs, sampling_params=sampling_params)
    for i, output in enumerate(outputs):
        generated_text = output.outputs[0].text
        print()
        print('=' * 40)
        print(f"Generated text: {generated_text!r}")

video_kwargs: {'do_sample_frames': False}
INFO 12-10 14:24:58 [utils.py:253] non-default args: {'trust_remote_code': True, 'max_model_len': 32768, 'gpu_memory_utilization': 0.5, 'disable_log_stats': True, 'model': '/mnt/SSD2_8TB/zechuan/huggingface/models--Qwen--Qwen3-VL-4B-Instruct/snapshots/ebb281ec70b05090aa6165b016eac8ec08e71b17'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 12-10 14:24:58 [model.py:637] Resolved architecture: Qwen3VLForConditionalGeneration
INFO 12-10 14:24:58 [model.py:1750] Using max model len 32768
INFO 12-10 14:24:58 [scheduler.py:228] Chunked prefill is enabled with max_num_batched_tokens=8192.


/home/zechuan/miniconda3/envs/vllm/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


(EngineCore_DP0 pid=2531099) INFO 12-10 14:25:02 [core.py:93] Initializing a V1 LLM engine (v0.12.0) with config: model='/mnt/SSD2_8TB/zechuan/huggingface/models--Qwen--Qwen3-VL-4B-Instruct/snapshots/ebb281ec70b05090aa6165b016eac8ec08e71b17', speculative_config=None, tokenizer='/mnt/SSD2_8TB/zechuan/huggingface/models--Qwen--Qwen3-VL-4B-Instruct/snapshots/ebb281ec70b05090aa6165b016eac8ec08e71b17', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=32768, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoni

...

[rank0]:[W1210 14:25:33.876244391 ProcessGroupNCCL.cpp:1524] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())



Inputs[0]: input_['prompt']="<|im_start|>user\n<|vision_start|><|image_pad|><|vision_end|>图片如果包含任何文本或表格数据，生成简短的描述，概述其关键信息；如果没有，请返回 '无描述'。<|im_end|>\n<|im_start|>assistant\n"

>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Generated text: '该图表展示了2021至2024年“脱碳路径”中每收入单位的温室气体排放量及减排量。绿色折线显示每单位收入温室气体排放量从10.5降至9.2（单位：百万吨二氧化碳当量/百万元人民币）；柱状图分为“范围一”（深色）和“范围二、三”（浅色）排放量，2024年总排放量为9.2，显示减排趋势。'


In [11]:

if __name__ == '__main__':
    prompt = "图片如果包含任何文本或表格数据，生成简短的描述，概述其关键信息；如果没有，请返回 '无描述'。"
    messages = [
        {
            "role": "user",
            "content": [
              {
                  "type": "image",
                  "image": "/home/zechuan/m3docrag/contents/visdom_test_output_text/dots_ocr_text_test/2024_Tencent_ESG/2024_Tencent_ESG_page_94_img_21.png",
              },
              {"type": "text", "text": prompt},
            ],
        }
    ]

    # TODO: change to your own checkpoint path
    checkpoint_path = "/mnt/SSD2_8TB/zechuan/huggingface/models--Qwen--Qwen3-VL-4B-Instruct/snapshots/ebb281ec70b05090aa6165b016eac8ec08e71b17"
    processor = AutoProcessor.from_pretrained(checkpoint_path)
    inputs = [prepare_inputs_for_vllm(message, processor) for message in [messages]]

    llm = LLM(
        model=checkpoint_path,
        trust_remote_code=True,
        gpu_memory_utilization=0.5,
        enforce_eager=False,
        seed=0,
        max_model_len=32768
    )

    sampling_params = SamplingParams(
        temperature=0,
        max_tokens=1024,
        top_k=-1,
        stop_token_ids=[],
    )

    for i, input_ in enumerate(inputs):
        print()
        print('=' * 40)
        print(f"Inputs[{i}]: {input_['prompt']=!r}")
    print('\n' + '>' * 40)

    outputs = llm.generate(inputs, sampling_params=sampling_params)
    for i, output in enumerate(outputs):
        generated_text = output.outputs[0].text
        print()
        print('=' * 40)
        print(f"Generated text: {generated_text!r}")

video_kwargs: {'do_sample_frames': False}
INFO 12-10 14:31:00 [utils.py:253] non-default args: {'trust_remote_code': True, 'max_model_len': 32768, 'gpu_memory_utilization': 0.5, 'disable_log_stats': True, 'model': '/mnt/SSD2_8TB/zechuan/huggingface/models--Qwen--Qwen3-VL-4B-Instruct/snapshots/ebb281ec70b05090aa6165b016eac8ec08e71b17'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 12-10 14:31:00 [model.py:637] Resolved architecture: Qwen3VLForConditionalGeneration
INFO 12-10 14:31:00 [model.py:1750] Using max model len 32768
INFO 12-10 14:31:00 [scheduler.py:228] Chunked prefill is enabled with max_num_batched_tokens=8192.


/home/zechuan/miniconda3/envs/vllm/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


(EngineCore_DP0 pid=2535102) INFO 12-10 14:31:04 [core.py:93] Initializing a V1 LLM engine (v0.12.0) with config: model='/mnt/SSD2_8TB/zechuan/huggingface/models--Qwen--Qwen3-VL-4B-Instruct/snapshots/ebb281ec70b05090aa6165b016eac8ec08e71b17', speculative_config=None, tokenizer='/mnt/SSD2_8TB/zechuan/huggingface/models--Qwen--Qwen3-VL-4B-Instruct/snapshots/ebb281ec70b05090aa6165b016eac8ec08e71b17', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=32768, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoni

...

[rank0]:[W1210 14:31:35.043344075 ProcessGroupNCCL.cpp:1524] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())



Inputs[0]: input_['prompt']="<|im_start|>user\n<|vision_start|><|image_pad|><|vision_end|>图片如果包含任何文本或表格数据，生成简短的描述，概述其关键信息；如果没有，请返回 '无描述'。<|im_end|>\n<|im_start|>assistant\n"

>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Generated text: '无描述'
